## **AI Investment Advisor Chatbot**

### ข้อกำหนดของโปรเจกต์
- **Library สร้าง Agent**: LangChain + LangGraph
- **Embedding Model**: Google Gemini (text-embedding-004)
- **Vector Search**: ChromaDB
- **Tools**: สำหรับคำนวณผลตอบแทน, แนะนำการลงทุน
- **Logging**: บันทึก log การทำงานของ Agent
- **กรองคำถาม**: ถามอื่นที่ไม่เกี่ยวกับการลงทุนจะไม่ตอบ
- **Web Interface**: Gradio รันบน localhost

## **1. Installation**

In [ ]:
pip install google-genai python-dotenv gradio numpy chromadb langchain langchain-google-genai langgraph langchain-community

## **2. Setup API Key & Model**

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv()

# Set API Key
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "YOUR_API_KEY")

# Initialize Chat Model (LLM)
llm = init_chat_model("google_genai:gemini-2.0-flash")

# Initialize Embedding Model
embeddings = GoogleGenerativeAIEmbeddings(
    model="text-embedding-004"
)

print("✅ Model and Embedding initialized successfully!")

## **3. Setup Logging**

In [ ]:
import logging
from datetime import datetime

# Create logs directory
os.makedirs("logs", exist_ok=True)

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f'logs/agent_{datetime.now().strftime("%Y%m%d")}.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("🚀 Agent logging initialized!")

## **4. Setup Vector Search with ChromaDB**

In [ ]:
import chromadb
from chromadb.config import Settings
from langchain.schema import Document
from langchain_community.vectorstores import Chroma

# Investment knowledge base
investment_docs = [
    Document(
        page_content="การลงทุนในหุ้น (Stock Investment) คือการซื้อหุ้นของบริษัทจดทะเบียน เพื่อเป็นเจ้าของบริษัทและรับผลตอบแทนจากการเติบโตของบริษัท มีความเสี่ยงสูงแต่ผลตอบแทนในระยะยาวดี",
        metadata={"type": "stocks", "category": "equity"}
    ),
    Document(
        page_content="กองทุนรวม (Mutual Fund) คือการรวมเงินลงทุนของนักลงทุนหลายคน ให้ผู้จัดการกองทุนบริหารจัดการ มีความเสี่ยงต่ำกว่าการลงทุนในหุ้นโดยตรง",
        metadata={"type": "funds", "category": "managed"}
    ),
    Document(
        page_content="พันธบัตรรัฐบาล (Government Bond) คือการลงทุนในหนี้สินของรัฐบาล มีความเสี่ยงต่ำ ผลตอบแทนคงที่ เหมาะสำหรับนักลงทุนที่ต้องการความมั่นคง",
        metadata={"type": "bonds", "category": "fixed_income"}
    ),
    Document(
        page_content="การลงทุนในทองคำ (Gold Investment) เป็นสินทรัพย์ปลอดภัยในยามวิกฤต มักเคลื่อนไหวตรงกันข้ามกับตลาดหุ้น ควรถือเป็นส่วนหนึ่งของพอร์ตการลงทุน",
        metadata={"type": "gold", "category": "commodity"}
    ),
    Document(
        page_content="การจัดพอร์ตการลงทุน (Asset Allocation) ควรกระจายความเสี่ยง เช่น 60% หุ้น 40% ตราสารหนี้ ขึ้นอยู่กับความเสี่ยงที่รับได้",
        metadata={"type": "strategy", "category": "portfolio"}
    ),
    Document(
        page_content="คำแนะนำสำหรับมือใหม่: เริ่มต้นลงทุนในกองทุนรวมดัชนี (Index Fund) ที่มีค่าธรรมเนียมต่ำ กระจายความเสี่ยงด้วยการซื้อสม่ำเสมอ (Dollar Cost Averaging)",
        metadata={"type": "advice", "category": "beginner"}
    ),
    Document(
        page_content="หลักการลงทุน: ลงทุนในสิ่งที่เข้าใจ กระจายความเสี่ยง ถือระยะยาว ไม่ลงทุนด้วยเงินที่ต้องการใช้เร็ว",
        metadata={"type": "principle", "category": "strategy"}
    ),
    Document(
        page_content="การลงทุนในบริษัทที่มีปันผลสูง (Dividend Stocks) เหมาะสำหรับผู้ต้องการรายได้ประจำ มีความเสี่ยงปานกลาง",
        metadata={"type": "dividend", "category": "equity"}
    )
]

# Create ChromaDB vector store
vectorstore = Chroma.from_documents(
    documents=investment_docs,
    embedding=embeddings,
    collection_name="investment_knowledge"
)

print(f"✅ ChromaDB initialized with {len(investment_docs)} documents!")
logger.info(f"Vector store created with {len(investment_docs)} documents")

## **5. Create Tools for Agent**

In [ ]:
from langchain.tools import tool
from langchain.tools import Tool
import math

@tool
def calculate_investment_return(
    principal: float,
    rate: float,
    years: float
) -> float:
    """คำนวณผลตอบแทนจากการลงทุนแบบทบต้น (Compound Interest)
    
    Args:
        principal: เงินลงทุนเริ่มต้น (บาท)
        rate: อัตราผลตอบแทนต่อปี (%)
        years: จำนวนปีที่ลงทุน
    
    Returns:
        จำนวนเงินรวมที่ได้รับ
    """
    result = principal * math.pow(1 + rate/100, years)
    logger.info(f"Calculated return: principal={principal}, rate={rate}%, years={years}, result={result}")
    return round(result, 2)

@tool
def search_investment_info(query: str) -> str:
    """ค้นหาข้อมูลเกี่ยวกับการลงทุนจากฐานความรู้
    
    Args:
        query: คำถามเกี่ยวกับการลงทุน
    
    Returns:
        ข้อมูลที่เกี่ยวข้อง
    """
    docs = vectorstore.similarity_search(query, k=2)
    result = "\n\n".join([d.page_content for d in docs])
    logger.info(f"Searched for: {query}, found {len(docs)} documents")
    return result

@tool
def recommend_portfolio(risk_level: str) -> str:
    """แนะนำพอร์ตการลงทุนตามระดับความเสี่ยง
    
    Args:
        risk_level: ระดับความเสี่ยงที่รับได้ (low/medium/high)
    
    Returns:
        แนะนำพอร์ตการลงทุน
    """
    portfolios = {
        "low": "พอร์ตความเสี่ยงต่ำ: 60% พันธบัตร + 30% กองทุนรวมตราสารหนี้ + 10% หุ้นปันผล",
        "medium": "พอร์ตความเสี่ยงปานกลาง: 50% หุ้น + 30% กองทุนรวม + 20% ตราสารหนี้",
        "high": "พอร์ตความเสี่ยงสูง: 80% หุ้น + 15% กองทุนรวม + 5% ทองคำ"
    }
    result = portfolios.get(risk_level.lower(), "กรุณาระบุระดับความเสี่ยง: low, medium, หรือ high")
    logger.info(f"Recommended portfolio for risk level: {risk_level}")
    return result

# Create tool list
tools = [
    calculate_investment_return,
    search_investment_info,
    recommend_portfolio
]

print("✅ Tools created successfully!")
logger.info("Tools: calculate_investment_return, search_investment_info, recommend_portfolio")

## **6. Create Investment Advisor Agent**

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# System prompt for investment advisor
system_prompt = """
คุณเป็นที่ปรึกษาการลงทุนที่มีประสบการณ์

ข้อกำหนด:
1. ตอบเฉพาะคำถามที่เกี่ยวกับการลงทุนเท่านั้น
2. ถ้าถามคำถามที่ไม่เกี่ยวกับการลงทุน ให้ตอบว่า "ขออภัย ฉันเป็นที่ปรึกษาการลงทุน สามารถช่วยตอบคำถามเกี่ยวกับการลงทุนเท่านั้น"
3. ใช้ tools ที่มีให้เพื่อคำนวณและค้นหาข้อมูล
4. ตอบเป็นภาษาไทย
5. ถ้าต้องการคำนวณผลตอบแทน ให้ใช้ calculate_investment_return
6. ถ้าต้องการข้อมูลเกี่ยวกับการลงทุน ให้ใช้ search_investment_info
7. ถ้าต้องการแนะนำพอร์ตการลงทุน ให้ใช้ recommend_portfolio
8. บันทึกการทำงานทุกครั้ง
"""

# Create agent
investment_agent = create_agent(
    model="google_genai:gemini-2.0-flash",
    tools=tools,
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

print("✅ Investment Advisor Agent created!")
logger.info("Investment Agent initialized")

## **7. Test Agent**

In [ ]:
# Test 1: Ask about investment
response = investment_agent.invoke(
    {"messages": [{"role": "user", "content": "ถ้าลงทุน 100,000 บาท อัตราผลตอบแทน 7% ต่อปี ระยะเวลา 10 ปี จะได้เงินเท่าไหร่?"}]},
    {"configurable": {"thread_id": "test1"}}
)

print("Test 1: คำนวณผลตอบแทน")
print("=" * 50)
for msg in response['messages']:
    if hasattr(msg, 'content') and msg.content:
        print(f"{msg.type}: {msg.content}")
        print("-" * 30)

In [ ]:
# Test 2: Search investment info
response2 = investment_agent.invoke(
    {"messages": [{"role": "user", "content": "การลงทุนในหุ้นเหมาะกับใคร?"}]},
    {"configurable": {"thread_id": "test2"}}
)

print("Test 2: ค้นหาข้อมูลการลงทุน")
print("=" * 50)
for msg in response2['messages']:
    if hasattr(msg, 'content') and msg.content:
        print(f"{msg.type}: {msg.content}")
        print("-" * 30)

In [ ]:
# Test 3: Non-investment question
response3 = investment_agent.invoke(
    {"messages": [{"role": "user", "content": "วันนี้อากาศเป็นอย่างไร?"}]},
    {"configurable": {"thread_id": "test3"}}
)

print("Test 3: คำถามที่ไม่เกี่ยวกับการลงทุน")
print("=" * 50)
for msg in response3['messages']:
    if hasattr(msg, 'content') and msg.content:
        print(f"{msg.type}: {msg.content}")
        print("-" * 30)

## **8. Run Gradio Web Interface**

In [ ]:
import gradio as gr

def chat(message, history):
    """Chat function for Gradio"""
    logger.info(f"User message: {message}")
    
    # Generate unique thread ID for each conversation
    import uuid
    thread_id = str(uuid.uuid4())
    
    try:
        response = investment_agent.invoke(
            {"messages": [{"role": "user", "content": message}]},
            {"configurable": {"thread_id": thread_id}}
        )
        
        # Get last AI message
        ai_message = ""
        for msg in reversed(response['messages']):
            if hasattr(msg, 'type') and msg.type == 'ai':
                ai_message = msg.content
                break
        
        logger.info(f"Agent response: {ai_message[:100]}...")
        return ai_message
    
except Exception as e:
    error_msg = f"เกิดข้อผิดพลาด: {str(e)}"
    logger.error(f"Error: {str(e)}")
    return error_msg

# Create Gradio interface
demo = gr.ChatInterface(
    fn=chat,
    title="🤖 AI ที่ปรึกษาการลงทุน",
    description="ถามคำถามเกี่ยวกับการลงทุนได้เลย (หุ้น, กองทุน, พันธบัตร, กองทุนรวม ฯลฯ)",
    theme=gr.themes.Soft(),
    examples=[
        ["ถ้าลงทุน 50,000 บาท อัตรา 8% ต่อปี 5 ปี จะได้เท่าไหร่?"],
        ["การลงทุนในกองทุนรวมเหมาะกับใคร?"],
        ["แนะนำพอร์ตการลงทุนสำหรับคนที่รับความเสี่ยงได้ปานกลาง"],
        ["หลักการลงทุนที่ดีมีอะไรบ้าง?"]
    ]
)

# Launch Gradio
print("=" * 50)
print("🚀 Starting Gradio Web Interface...")
print("📍 URL: http://localhost:7860")
print("=" * 50)

demo.launch(server_name="localhost", server_port=7860)

## **สรุปข้อกำหนดของโปรเจกต์**

| รายการ | รายละเอียด |
|--------|------------|
| **Library สร้าง Agent** | LangChain + LangGraph |
| **Embedding Model** | Google Gemini (text-embedding-004) |
| **Vector Search** | ChromaDB |
| **Tools** | calculate_investment_return, search_investment_info, recommend_portfolio |
| **Logging** | บันทึกในไฟล์ logs/agent_YYYYMMDD.log |
| **Web Interface** | Gradio (localhost:7860) |
| **กรองคำถาม** | ปฏิเสธคำถามที่ไม่เกี่ยวกับการลงทุน |